In [2]:
from similarity.cosine_similarity import similarity, similarities
import requests
import wikipedia
from bs4 import BeautifulSoup

In [6]:
sr = wikipedia.search("Lewis Gilbert (American_football)", results=10, suggestion=False)
print(sr)
sr_sugg = wikipedia.search("Lewis Gilbert (American_football)", results=10, suggestion=True)
print(sr_sugg)

['Lewis Gilbert (American football)', 'Lewis Gilbert (disambiguation)', 'Sean Lewis (American football)', 'John Lewis (disambiguation)', 'Sean Gilbert', '1963 Lewis & Clark Pioneers football team', 'Gilbert (surname)', 'Dan Gilbert', 'Gilbert Arenas', 'Gilbert, Arizona']
(['Lewis Gilbert (American football)', 'Lewis Gilbert (disambiguation)', 'Sean Lewis (American football)', 'John Lewis (disambiguation)', 'Sean Gilbert', 'Gilbert (surname)', '1963 Lewis & Clark Pioneers football team', 'Dan Gilbert', 'Gilbert Arenas', 'Gilbert, Arizona'], 'lewis gilbert american football')


In [23]:
page_ = wikipedia.page("Adrestia", redirect=False, auto_suggest=False)
print(page_.title)
print(page_.summary)

RedirectError: "Adrestia" resulted in a redirect. Set the redirect property to True to allow automatic redirects.

In [20]:
query_params = {
        'prop': 'revisions',
        'rvprop': 'content',
        'rvparse': '',
        'rvlimit': 1
      }
query_params['titles'] = "WarGames"

page = wikipedia.WikipediaPage("WarGames")
page.title

'WarGames'

In [25]:
from utils.wiki_helper import get_exact_page_from_entity

In [ ]:
def find_gt_passage(wiki_data_entity: str, prompt: str):
    if wiki_data_entity.startswith("https://en.wikipedia.org/"):
        page_url = wiki_data_entity
    else:
        # Fetch Wikipedia page details (summary, content, and URL)
        page = get_exact_page_from_entity(wiki_data_entity)
        title = page.title
        summary = page.summary
        content = page.content
        url = page.url

        page_text = f"Title: {title}\n{content[:4000]}"

        page_data = {
            "title": title,
            "context": page_text,
        }
        
        # Step 5: Embed and compute similarity
        page_embedding = self.embedding_adapter.encode(page_text)
        prompt_embedding = self.embedding_adapter.encode(prompt)
        sim_score = similarity(page_embedding, prompt_embedding)
        return [(page_data, sim_score)]
    

In [ ]:
res = find_gt_passage("Q4217252", "who is thecomposer of WarGames")
print(res)
res_2 = find_gt_passage("https://en.wikipedia.org/wiki/Adrestia", "who is thecomposer of WarGames")
print(res_2)

TypeError: find_gt_passage() got an unexpected keyword argument 'redirect'

In [24]:
! mv /vol/bitbucket/lst20/lex-eval_dataset/PopQA/shuffled.csv /vol/bitbucket/lst20/lex-eval_dataset/PopQA/shuffled_300.csv

In [26]:
! cd /vol/bitbucket/lst20/lex-eval_dataset/PopQA/ && ls

shuffled_300.csv  shuffled_COPY.csv  test.csv


In [2]:
import pandas as pd

In [15]:
old_df = pd.read_csv("/vol/bitbucket/lst20/lex-eval_dataset/PopQA/shuffled_300.csv")
print(len(old_df))

df = pd.read_csv("hf://datasets/akariasai/PopQA/test.tsv", sep="\t")

286


In [16]:
print(len(df))
print(old_df.columns)
df["original_index"] = df.index
print(df.columns)

14267
Index(['id', 'subj', 'prop', 'obj', 'subj_id', 'prop_id', 'obj_id',
       's_aliases', 'o_aliases', 's_uri', 'o_uri', 's_wiki_title',
       'o_wiki_title', 's_pop', 'o_pop', 'question', 'possible_answers',
       'original_index'],
      dtype='object')
Index(['id', 'subj', 'prop', 'obj', 'subj_id', 'prop_id', 'obj_id',
       's_aliases', 'o_aliases', 's_uri', 'o_uri', 's_wiki_title',
       'o_wiki_title', 's_pop', 'o_pop', 'question', 'possible_answers',
       'original_index'],
      dtype='object')


In [17]:
to_drop = old_df['original_index'].tolist()
df = df.drop(index=to_drop, errors='ignore')

In [18]:
print(len(old_df))

286


In [19]:
size = 1000 - len(old_df)
size_ratio = size / len(df)
print(size_ratio, size)


0.051069308347042416 714


In [20]:
from sklearn.model_selection import train_test_split
import utils.constants as constants

_, df_shuffled, _, _ = train_test_split(
            df,
            df["prop"],
            test_size=size_ratio,
            shuffle=True,
            random_state=constants.SEED,
            stratify=df["prop"],
        )

In [21]:
print(len(df_shuffled))
print(df_shuffled.columns)

714
Index(['id', 'subj', 'prop', 'obj', 'subj_id', 'prop_id', 'obj_id',
       's_aliases', 'o_aliases', 's_uri', 'o_uri', 's_wiki_title',
       'o_wiki_title', 's_pop', 'o_pop', 'question', 'possible_answers',
       'original_index'],
      dtype='object')


In [22]:
old_df = pd.concat([old_df, df_shuffled], axis=0, ignore_index=True)
print(len(old_df))

1000


In [23]:
old_df.to_csv("/vol/bitbucket/lst20/lex-eval_dataset/PopQA/shuffled.csv", index=False)

In [28]:
old_df.iloc[288]


id                                                         2859281
subj                                         The Girl from Nowhere
prop                                                      producer
obj                                           Jean-Claude Brisseau
subj_id                                                    1225264
prop_id                                                        164
obj_id                                                     2914379
s_aliases           ["Girl from Nowhere","La Fille de nulle part"]
o_aliases                                                       []
s_uri                        http://www.wikidata.org/entity/Q38235
o_uri                       http://www.wikidata.org/entity/Q953105
s_wiki_title                     The Girl from Nowhere (2012 film)
o_wiki_title                                  Jean-Claude Brisseau
s_pop                                                         1555
o_pop                                                         

In [ ]:
import numpy as np
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx
import matplotlib.pyplot as plt
import concurrent.futures
import torch
import wikipedia

from collections import deque
from typing import List
import random
import time
import logging

from utils.wiki_helper import WikiHelper, get_exact_page_from_entity
class RootNode:
    def __init__(self, title: str):
        self.title = title
        self.children = []
        self.visited = set() # set of entities
        
    def add_child(self, node):
        self.children.append(node)
    
    def visualize(self):
        # Initialize a directed graph
        G = nx.DiGraph()

        # Recursive function to traverse nodes and add them to the graph
        def add_node_edges(node, parent_label=None):
            # Create a label based on the type of node
            if isinstance(node, RootNode):
                node_label = f"Root: {node.title}"
            else:
                node_label = f"Page: {node.title}"
            
            G.add_node(node_label)
            
            if parent_label:
                G.add_edge(parent_label, node_label)
            
            # Recursively process all children
            for child in node.children:
                add_node_edges(child, node_label)

        # Start building the graph from the root
        add_node_edges(self)

        # Draw the graph using a spring layout
        pos = nx.spring_layout(G)
        plt.figure(figsize=(8, 6))
        nx.draw(G, pos, with_labels=True, node_color='lightblue', arrows=True, node_size=800, font_size=5)
        plt.title("Graph Visualization Using NetworkX")
        plt.show()
   
class PageNode:
    def __init__(self, title: str, summary: str, content: str, url: str):
        self.title = title
        self.summary = summary
        self.content = content
        self.url = url
        self.children = []
        
        # passsage to insert: list of tuples if (doc, tfidf score)
        self.document_list = []
    
    def add_child(self, node):
        self.children.append(node)

class KnowledgeGraph: 
    '''
    each knowledge graph is created per root prompt and stored there
    '''
    def __init__(self, entity: str, **kwargs):
        prompt: str = kwargs.get("prompt")
        k_hop, top_k = 2, 3
        self.embedder = kwargs.get("embedder")
        self.wiki_helper = WikiHelper(self.embedder)
        self.graph = self.create_graph(entity, k_hop, top_k)
        self.ordered_docs = self.get_unordered_docs(prompt=prompt, embedder=self.embedder)
        # Initialize a boolean array for tracking visited documents:
        self.visited_doc_flags = [False] * len(self.ordered_docs)
        
            
    def link_importance(self, title: str, content: str,  summary: str) -> int:
        """Returns an importance score for a link based on its presence in the text"""
        score = 0
        title_lower = title.lower()
        content_lower = content.lower()
        summary_lower = content.lower()
        score += content_lower.count(title_lower)
        score += (summary_lower.count(title_lower) * 2)
        return score


    def rank_sentences_by_tfidf(self, article: str) -> List[str]:
        # Split on newline, tokenize each non-empty line, and flatten
        raw_lines = article.split('\n')
        sentences = []
        for line in raw_lines:
            line = line.strip()
            if not line:
                continue
            for sent in sent_tokenize(line):
                # 2) Discard any sentence shorter than 5 characters
                if len(sent) >= 5:
                    sentences.append(sent)
        vectorizer = TfidfVectorizer()
        # Compute TF-IDF matrix
        tfidf_matrix = vectorizer.fit_transform(sentences)
        # Compute the average TF-IDF score for each sentence https://aclanthology.org/P04-1049.pdf
        sentence_scores = np.sum(tfidf_matrix.toarray(), axis=1)  
        # Sort sentences by their scores in descending order
        ranked_sentences = [(score, sentence) for score, sentence in sorted(zip(sentence_scores, sentences), reverse=True)]
        
        return ranked_sentences


    def get_k_documents(self, corpus: List[str], top_k: int = 50) -> List[str]:
        # use rank_sentences_by_tfidf to get top_k doc
        sentences = self.rank_sentences_by_tfidf(corpus)
        top_k_ret = min(len(sentences), top_k)
        return sentences[:top_k_ret]


    def create_graph(self, start_entity: str, k_hop: int, top_k: int):
        '''
        Create a graph/tree using BFS from the start_entity using Wikipedia links.
        '''
        start_time = time.time()
        
        # 1. Initialize root node with the start entity
        root = RootNode(start_entity)
        root.visited = {start_entity}
        queue = deque()
        
        # 2. Fetch the Wikipedia page for the start entity
        wiki_page = get_exact_page_from_entity(start_entity)
        
        node = PageNode(wiki_page.title,
                        wiki_page.summary,
                        wiki_page.content,
                        wiki_page.url)
        node.document_list = self.get_k_documents(node.content)
        root.add_child(node)
        root.visited.add(wiki_page.title)
        page_dict = {
            "title":wiki_page.title,
            "summary":wiki_page.summary,
            "content":wiki_page.content,
            "url":wiki_page.url,
            'links': wiki_page.links,
            }
        queue.append((node, page_dict))
        
        # 3. BFS expansion
        for i in range(k_hop):
            to_process = deque()
            while queue:
                parent_node, parent_page = queue.popleft()
                try:
                    links = list(set(parent_page['links']))

                except wikipedia.exceptions.DisambiguationError:
                    logging.error("create_graph: wikipedia.exceptions.DisambiguationError: no links?")
                link_weights = []
                valid_links = []
                for link in links:
                    if link not in root.visited:
                        try:
                            weight = self.link_importance(link, parent_page['content'],  parent_page['summary'])
                            link_weights.append(weight)
                            valid_links.append(link)
                        except Exception as e:
                            logging.error("create_graph: Exception weights", e)
                
                # Sort valid_links based on weights and select the top_k links.
                sorted_indices = sorted(range(len(link_weights)), key=lambda i: link_weights[i])
                top_indices = sorted_indices[:min(len(valid_links), top_k)]  # Use valid_links length here.
                top_links = [valid_links[i] for i in top_indices]
                with concurrent.futures.ThreadPoolExecutor() as executor:
                    futures = {executor.submit(self.wiki_helper.fetch_wiki_page_with_retry, link): link for link in top_links}
                    for future in concurrent.futures.as_completed(futures):
                        child_page = future.result()
                        if not child_page:
                            continue  # Skip pages that couldn't be fetched or are ambiguous.
                        child_node = PageNode(child_page["title"], child_page["summary"], child_page["content"], child_page["url"])
                        # Generate passages (passage, tfidf score)
                        child_node.document_list = self.get_k_documents(child_page["content"])
                        parent_node.add_child(child_node)
                        to_process.append((child_node, child_page))
                        
                for tl in top_links:
                    root.visited.add(tl)
                        
            queue = to_process

        logging.info(f"--- create graph: {time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))} ---")
        return root

        
    def get_unordered_docs(self, **kwargs):
        """
        Proposes next documents for the perturber to add to the prompt.
        Documents are collected via a breadth-first traversal of the document tree,
        and returned in the order they are encountered (no similarity-based sorting).

        Returns:
            List[Tuple[str, str, str]]:
                A list of tuples each containing:
                - Document text,
                - Document title, and
                - Document content.
        """
        import time
        from collections import deque

        start_time = time.time()
        doc_db = []
        title_db = []
        content_db = []
        
        # Get the set of titles from the root's direct children to avoid collecting their document lists.
        direct_children_titles = {child.title for child in self.graph.children}
        logging.info("direct_children_titles", direct_children_titles)
        queue = deque([self.graph])
        
        while queue:
            node = queue.popleft()
            # If node is a PageNode and not a direct child of the root,
            # collect its document sentences.
            if isinstance(node, PageNode) and node.title not in direct_children_titles:
                for entry in node.document_list:
                    # Each entry is assumed to be a tuple (score, sentence); we use the sentence.
                    doc_db.append(entry[1])
                    title_db.append(node.title)
                    content_db.append(node.content)
            # Add child nodes to the queue for further traversal.
            for child in node.children:
                queue.append(child)
        
        unordered_docs = [(doc_db[i], title_db[i], content_db[i]) for i in range(len(doc_db))]
        logging.info(f"--- get_unordered_docs: {time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))} seconds ---")
        return unordered_docs

    def get_next_document(self):
        """
        Proposes the next document for the perturber to add to the prompt that has not been visited.
        """
        # Identify indices for documents that have not been visited.
        available_indices = [i for i, visited in enumerate(self.visited_doc_flags) if not visited]
        if not available_indices:
            raise RuntimeError(f"All documents are visited. doclist length = {len(self.ordered_docs)}")
        # Randomly select an available document's index.
        selected_index = random.choice(available_indices)
        doc, title, content = self.ordered_docs[selected_index]
        return (doc, title, content, selected_index)


    def update_visit_status(self, index: int):
        """
        Marks the document at the provided index as visited by setting the corresponding
        boolean flag to True
        """
        if self.visited_doc_flags[index]:
            raise RuntimeError(f"Document at index {index} has already been visited.")
        self.visited_doc_flags[index] = True


In [22]:
kg = KnowledgeGraph("Yorushika")
kg.ordered_docs

create_graph root pages: Yorushika
create_graph: Yorushika links: 72
top_links ['DVD', 'J-pop', 'Snake (Yorushika song)']


ERROR:root:wikipedia.exceptions.PageError for 'DVD'.


create_graph: Snake (Yorushika song) links: 29
top_links ['Sunny (Yorushika song)', 'Ghost in a Flower', "That's Why I Gave Up on Music"]
create_graph: C-pop links: 277
top_links ['Urban adult contemporary', 'Worldbeat', 'Lee Yee']


ERROR:root:wikipedia.exceptions.PageError for 'Lee Yee'.


--- create graph: 00:00:05 ---
direct_children_titles {'Yorushika'}
Yorushika
Yorushika
Snake (Yorushika song)
C-pop
Ghost in a Flower
That's Why I Gave Up on Music
Sunny (Yorushika song)
Urban adult contemporary
Worldbeat
--- get_unordered_docs: 00:00:00 seconds ---


[('In the music magazine ROCKIN\'ON JAPAN (March 2025 issue), writer Mie Sugiura mentioned that this work is based on a stanza of Won Chung \'s poem, and wondered how far n-buna\'s "knowledge" extends, and could only be envious of his rich literary knowledge.',
  'Snake (Yorushika song)',
  '"Snake" (へび, Hebi) is a song by Japanese rock band Yorushika. It was released as a digital single by Polydor Records on January 17, 2025. The song, inspired by a passage from "Li-Sui" (Di-Thoughts) by the Tang Dynasty poet Yuan Zhen, was used as the ending theme for the NHK General TV anime series Chi: On the Movement of the Earth.\n\n\n== Background ==\nOn January 8, 2025, it was announced on the official X that the new ending theme for the TV anime Chi. About the Earth\'s Movement would be "Snake". This song was used from the 16th episode that aired on the 11th of the same month . n-buna explained that one day he saw a snake with beautiful scales, and felt an affinity with the anime when he was w